In [ ]:
from dotenv import load_dotenv
load_dotenv()

from typing_extensions import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

#SHORT TERM MEMORY persistence (save conversation state/message history/thread checkpoints)
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.postgres import PostgresSaver

#🔥🔥LONG TERM memory persistence (save ANCHE facts/preferences/profile/semantic memories)
from langgraph.store.postgres import PostgresStore  #🔥
from langgraph.store.sqlite import SqliteStore  #🔥

from langchain_ollama import ChatOllama, OllamaEmbeddings  #OllamaEmbeddings è il wrapper di langchain x usare i modelli Ollama come EMBEDDING MODEL(cioe models usati solo x 'tagliuzzare'), che puoi usare x creare vettori/embeddings da salvare nel tuo long term memory store (PostgresStore/SqliteStore) e poi fare ricerche semantiche su quei vettori/embeddings per recuperare informazioni rilevanti da usare nelle tue chat con l'agente.
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
import psycopg  #postgresql moderno per python, più facile da usare di psycopg2, supporta async, ect
import os

# Configuration
BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen3"
EMBEDDING_MODEL = "nomic-embed-text"

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=BASE_URL)

In [ ]:
embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL, 
    base_url=BASE_URL)
#crei emebedding engine

def embed_texts(texts: list[str]) -> list[list[float]]:
    return embeddings.embed_documents(texts)
#input: lista di str, output: lista di vettori (lista di lista di float), dove ogni vettore rappresenta l'embedding della corrispondente stringa di input. Questa funzione è quella che passerai al tuo PostgresStore/SqliteStore come funzione di embedding, così quando salvi dei dati nel store, langgraph sa come creare gli embedding da quei dati usando questa funzione.

db_url = os.getenv("POSTGRESQL_URL")  #get env var and save it in var

checkpointer_conn = psycopg.connect(
    db_url,  
    autocommit=True,   
    prepare_threshold=0)
checkpointer = PostgresSaver(checkpointer_conn)  #save as var, SHORTTERMMEMORY checkpointer, che userai x salvare checkpoint della conversazione/thread state, ect

store_conn = psycopg.connect(
    db_url, 
    autocommit=True, 
    prepare_threshold=0)
store = PostgresStore(store_conn, index = {'embed': embed_texts, 'dims': 768}) #save as var, LONGTERMMEMORY store  

checkpointer.setup()  #crea tabelle necessarie in postgresql per salvare checkpoint, 🔥da eseguire solo la prima volta, poi puoi commentare questa riga o rimuoverla, altrimenti ogni volta che esegui lo script cercherà di creare le tabelle e ti darà errore se esistono già. In produzione puoi gestire migrazioni e setup del db in modo più robusto con strumenti come Alembic o Flyway.
store.setup()  #crea tabelle longtermmemory necessarie in postgresql, questo comand solo da execute 1 volta

In [ ]:
#memory operations: PUT(), GET(), SEARCH(), DELETE()

user_id = "kgptalkie"
namespace = (user_id, "preferences")  #organizza memorie logicamente 

store.put(namespace, "food", {
    "diet": "veg", 
    "likes": ["pasta", "pizza", "veggies"]})
#namespace=dove salvare, memory key=food, {...} è il memory value

store.put(namespace, "color", {
    "favorite": "blue", 
    "dislike": "brown"})

store.put(namespace, "work", {
    "role": "Data Scientist",
    "interests": ["machine learning", "ai", "gen ai", "agents"]
})

store.get(namespace, "color")  #GET
store.delete(namespace, "color")  #DELETE

In [ ]:
query = "What does kgp talkie like to eat?"
results = store.search(namespace, query=query, limit=1)
results

query = "What does kgp talkie like in colors?"
results = store.search(namespace, query=query, limit=1)
results

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    #grazie ad Annotated, puoi concatenare! FONDAMENTALE
    #e.g
    #{  
    #   "messages": [msg1]  #stato old in input
    # }
    #{  
    #   "messages": [msg1]  #stato updated
    # }
    #e ottieni merge grazie a operator.add
    # {
    #   "messages": [msg1, msg2]
    # }
    #quindi in questo modo ottieni la history
    user_id: str

In [ ]:
@tool
def save_user_memory(user_id:str, category:str, information:dict) -> str:
    """
    Save user preference or information to long-term memory.

    Args:
        user_id: User identifier
        category: Category of information (e.g., 'food', 'work', 'hobbies', 'schedule', 'location')
        information: Dictionary containing the information to save
    """
    namespace = (user_id, "preferences")
    store.put(namespace, category, information)  #save in memory
    return f"Saved {category} preferences."

@tool
def get_user_memory(user_id:str, category:str) -> str:
    """
        Retrieve user preference or information from long-term memory.

        Args:
            user_id: User identifier
            category: Category of information to retrieve (e.g., 'food', 'work', 'hobbies')
        """
    namespace = (user_id, "preferences")
    item = store.get(namespace, category)
    if item:
        return f"{category}: {item.value}"
    else:
        return f"No '{category}' information found!"

# if you want to use store.search
# @tool
# def get_user_memory(user_id:str, query:str) -> str:
# store.search(namespace, query)

In [ ]:

get_user_memory.invoke({"user_id": 'kgptalkie', "category": 'color'})
get_user_memory.invoke({"user_id": 'kgptalkie', "category": 'something'})

In [ ]:
import sys
sys.path.append("../05. LangGraph ReAct Agent with Tools")
import my_tools

# my_tools.get_weather.invoke({'location': "Mumbai"})

my_tools.calculate.invoke({'expression': '2+2*1.4/23-34'})
all_tools = [my_tools.get_weather, my_tools.calculate]  #2 tools utilizzabili dall'agente, questi 2 tools non lavorano su memory

In [ ]:
def agent_node(state: AgentState):
    store_conn = psycopg.connect(
        db_url, 
        autocommit=True, 
        prepare_threshold=0)
    store = PostgresStore(store_conn, index = {'embed': embed_texts, 'dims': 768}) #save as var, LONGTERMMEMORY store

    user_id = state.get("user_id", "unknown")  #legge user_id dallo stato, 'unknown' as fallback
    namespace = (user_id, "preferences")

    last_message = state['messages'][-1].content  #get last mex
    memories = store.search(
        namespace, 
        query=last_message, 
        limit=3)
    #fa semantic memory retrieval (è tipo un'human-like memory retrieval! TROVA MEMORIES SEMANTICAMENTE RILEVANTI avendo l'ultimo mex) basata sull'ultimo messaggio dell'utente, così recupera le memorie più rilevanti per quel messaggio, che poi userai come contesto nella risposta dell'agente. In questo modo l'agente può ricordare informazioni passate e usarle in modo intelligente nelle conversazioni future, rendendo l'interazione più personalizzata e contestuale.

    # build context memory for personalized answer
    context_line = []  #conterra memory snippets
    for mem in memories:
        text = f" -{mem.key}: {mem.value}"  #e.g. -food: {'likes': ['pizza']}, -work: {'role':'engineer'}
        context_line.append(text)

    memory_text = "\n\n".join(context_line) if context_line else "No user preferences stored yet!"  #concatena tutto in un unico testo da passare al sistema prompt, così l'agente ha tutto il contesto delle memorie rilevanti in un formato leggibile, che può usare per generare risposte più informate e personalizzate. Se non ci sono memorie rilevanti, fornisce un messaggio di fallback.

    print(f"User Memory Retrieved: \n{memory_text}\n")
    tools = all_tools + [save_user_memory, get_user_memory]  #aggiungi questi 2 tools alle utility tools che già hai
    llm_with_tools = llm.bind_tools(tools)  #crea una versione del tuo llm che ha accesso a tutti i tools (ora sono 4)
    system_prompt = SystemMessage(f"""
        You are a helpful assistant with long-term memory capabilities and access to utility tools.

        User ID: {user_id}
        Current User Memories:{memory_text}

        MEMORY TOOLS USAGE:

        1. save_user_memory: Use when user shares NEW information
        - Always pass user_id: "{user_id}"
        - Food preferences (diet, likes, dislikes, allergies)
        - Work information (role, company, interests)
        - Hobbies and activities
        - Schedule and availability
        - Location and timezone

        2. get_user_memory: Use when you need to recall specific category
        - Always pass user_id: "{user_id}"
        - When answering questions about past preferences
        - When user asks "what do you know about me?"
        - When making recommendations based on preferences

        UTILITY TOOLS USAGE:
        3. get_weather: Use to retrieve current weather information
        - Pass location as parameter (city name, zip code, or coordinates)
        - Use when user asks about weather conditions
        - Use when planning activities that depend on weather
        - Examples: "What's the weather in London?", "Will it rain today?"

        4. calculate: Use to perform mathematical calculations
        - Pass mathematical expression as string parameter
        - Supports basic arithmetic (+, -, *, /)
        - Supports advanced operations (powers, roots, trigonometry)
        - Use when user needs numerical computations
        - Examples: "What's 15% of 250?", "Calculate the area of a circle with radius 5"

        GUIDELINES:
        - Always save when user shares personal information
        - Retrieve specific categories when needed for context
        - Use semantic search results shown above for general context
        - Use get_weather when location-based weather info is needed
        - Use calculate for any mathematical operations or conversions
        - Be conversational and natural when using all tools
        - Combine tools when appropriate (e.g., weather + saved location preference)
    """)
    
    messages = [system_prompt] + state['messages']  #concatena (update) 
    response = llm_with_tools.invoke(messages)
    if hasattr(response, 'tool_calls') and response.tool_calls:
        for tc in response.tool_calls:
            print(f"[AGENT] called Tool {tc.get('name', '?')} with args {tc.get('args', '?')}")
    else:
        print(f"[AGENT] Responding...")

    return {'messages': [response]}

In [ ]:
def should_continue(state: AgentState):
    last = state['messages'][-1]
    if hasattr(last, 'tool_calls') and last.tool_calls:
        return "tools"
    else:
        return END

In [ ]:
from langgraph.prebuilt import ToolNode

tools = all_tools + [save_user_memory, get_user_memory]  #prima l'avevi gia fatto, ma era dentro l'agente

def create_agent():
    builder = StateGraph(AgentState)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", should_continue, ["tools", END])
    builder.add_edge("tools", "agent")
    checkpointer_conn = psycopg.connect(
        db_url, 
        autocommit=True, 
        prepare_threshold=0)
    checkpointer = PostgresSaver(checkpointer_conn)  #save as var, SHORTTERMMEMORY store
    graph = builder.compile(checkpointer=checkpointer)

    return graph

In [ ]:
agent = create_agent()
agent

In [ ]:
user_id = "kgptalkie"
config = {'configurable': {'thread_id': f"{user_id}_longterm"}}
query = "I am Laxmi kant tiwari. I love AI and machine learning in python."
result = agent.invoke({'messages': [HumanMessage(query)],
                       'user_id': user_id}, 
                       config=config)
result['messages'][-1].pretty_print()


user_id = "kgptalkie"
config = {'configurable': {'thread_id': f"{user_id}_longterm"}}
query = "Could you please suggest me some diet plan?"
result = agent.invoke({'messages': [HumanMessage(query)],
                       'user_id': user_id}, config=config)
result['messages'][-1].pretty_print()


user_id = "kgptalkie"
config = {'configurable': {'thread_id': f"{user_id}_this_is_another_test"}}
query = "Could you please suggest me some diet plan?"
result = agent.invoke({'messages': [HumanMessage(query)],
                       'user_id': user_id}, 
                       config=config)
result['messages'][-1].pretty_print()